# 08 — Referral inflow and completion outflow (Part_3, Part_1A/1B)

The waiting list is a stock, and the raw files carry its flows — unused until now: `Part_3` (new RTT periods = referrals entering the list) and `Part_1A`/`Part_1B` (completed admitted/non-admitted pathways = patients leaving via treatment). Mechanically, next month's list = this month's list + inflow − outflow − other removals. Persistence forecasts "net flow ≈ 0"; if the flows are themselves persistent and nonzero, knowing them should predict the change.

Checked on the real files before building: `Part_3` rows carry their count **only in `Total All`** — every `Gt ...` week-band column is empty (0% filled in both schema generations), so band-summing would silently produce zeros. `Part_1A/1B` fill both.

The bar is unchanged: naive persistence MAE 153 on the 1,407 FY 2024-25 validation rows (04, 07). Same cohort, same splits, asserted identical. The sealed FY 2025-26 stays untouched.

In [1]:
import pandas as pd                          # tables
import numpy as np                           # array arithmetic
import os
from datetime import datetime                # parse 'RTT-OCTOBER-2019' into a real date
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor   # gradient-boosted trees, NaN-native
from sklearn.inspection import permutation_importance        # shuffle-a-column importance

RTT = Path("../data/rtt_csv")
PROC = Path("../data/processed")

TEST_START = pd.Timestamp("2025-04-01")      # sealed test year - never read below (same as 04/07)
VAL_START = pd.Timestamp("2024-04-01")       # validation year FY 2024-25
COVID = pd.Timestamp("2020-03-01")           # regime break: first lockdown month
SEED = 42                                    # the one declared seed

USECOLS = ["Period", "Provider Org Code", "Treatment Function Code",
           "RTT Part Type", "Total All"]     # all a flows panel needs - keeps the 96 reads fast
print(len(os.listdir(RTT)), "raw files")

96 raw files


## Build the flows panel

One row per (month, provider, specialty): `referrals` = Part_3 `Total All`, `completed` = Part_1A + Part_1B `Total All`, summed over commissioners. `C_999` excluded as always (it is the sum of the real rows and would double every figure).

In [2]:
# Goal: read the flow part-types from all 96 files into one panel.
rows = []
for f in sorted(os.listdir(RTT)):
    d = pd.read_csv(RTT / f, usecols=USECOLS)
    d = d[d["RTT Part Type"].isin(["Part_3", "Part_1A", "Part_1B"])
          & (d["Treatment Function Code"] != "C_999")]
    month = datetime.strptime(d["Period"].iloc[0].removeprefix("RTT-"), "%B-%Y")
    d["kind"] = np.where(d["RTT Part Type"] == "Part_3", "referrals", "completed")
    g = (d.groupby(["Provider Org Code", "Treatment Function Code", "kind"])["Total All"]
          .sum().unstack())                  # one column per kind; providers filing only one
                                             # part type get NaN in the other - kept as NaN
    rows.append(g.reset_index().assign(month=month))

flows = pd.concat(rows, ignore_index=True).rename(columns={
    "Provider Org Code": "provider_code", "Treatment Function Code": "tfunc_code"})
flows = flows[["month", "provider_code", "tfunc_code", "referrals", "completed"]]

assert flows["month"].nunique() == 96, flows["month"].nunique()   # a file failing to parse would show here
assert not flows.duplicated(["month", "provider_code", "tfunc_code"]).any()
assert flows[["referrals", "completed"]].min().min() >= 0         # a negative count = corrupt parse

flows.to_csv(PROC / "rtt_flows_panel.csv", index=False)
print(flows.shape, "->", PROC / "rtt_flows_panel.csv")

(338265, 5) -> ..\data\processed\rtt_flows_panel.csv


## Does net flow explain the change? National evidence first

Before any model: at the England level, does referrals − completions during a month line up with how much the list actually moved that month? It will overshoot — patients also leave without treatment (discharges, DNAs, admin removals) — but the correlation tells us whether the signal is real.

In [3]:
# Goal: national monthly net flow vs the national month-to-month change in the list.
wait = pd.read_csv(PROC / "rtt_incomplete_panel.csv", parse_dates=["month"])
nat = wait.groupby("month")["total_waiting"].sum().to_frame("waiting")
nat[["referrals", "completed"]] = flows.groupby("month")[["referrals", "completed"]].sum()
nat["net_flow"] = nat["referrals"] - nat["completed"]
nat["delta"] = nat["waiting"].diff()         # change during the month the flows happened in

ok = nat["delta"].notna()                    # the first month has no previous month to diff against
print(f"corr(net flow, actual change) = {nat.loc[ok, 'net_flow'].corr(nat.loc[ok, 'delta']):.2f}   (n={int(ok.sum())} months)")
print(f"mean monthly net flow {nat['net_flow'].mean():,.0f} vs mean actual change {nat.loc[ok, 'delta'].mean():,.0f}")
print("- the gap is patients leaving without treatment (removals), which Part_1 does not count")

corr(net flow, actual change) = 0.74   (n=95 months)
mean monthly net flow 265,730 vs mean actual change 31,666
- the gap is patients leaving without treatment (removals), which Part_1 does not count


## Forecast test: same bar, same rows as 04 and 07

The 07 feature set plus four flow features: this month's `referrals`, `completed`, their difference `net_flow`, and its 3-month mean (`net_flow_3m`, smoother). Timing note: forecasting month t+1 from the end of month t, the flows *during* t exist in the data — the model uses them to proxy the flows during t+1.

**Publication-lag caveat (2026-08-01 audit):** "exists" is not "is public". NHS publishes each monthly RTT extract roughly 6–8 weeks after the month ends, so at the end of month t neither the month-t flows nor the month-t list is yet published. Every model here — persistence included — sees the same information set, so all comparisons in this project are internally fair; but a *deployed* forecast would run from the latest published month, which makes its effective horizon ~3 months for a t+1 target (the h=3 row of notebook 09's table approximates that error level).

In [4]:
# Goal: the 04/07 cohort with the 07 features plus the flow features.
panel = pd.read_csv(PROC / "rtt_kh03_canonical_panel.csv", parse_dates=["month"])
dev = panel[panel["month"] < TEST_START]

df = dev[(dev["tfunc_code"] == "C_110") & dev["fl_client"] & dev["is_public"]].copy()
df = df.sort_values(["provider_code", "month"])

g = df.groupby("provider_code")["total_waiting"]
df["target_delta"] = g.shift(-1) - df["total_waiting"]   # next month's list minus this month's
for k in (1, 2, 3):
    df[f"delta_{k}"] = df["total_waiting"] - g.shift(k)  # momentum, as in 07
df["month_of_year"] = df["month"].dt.month

df = df.merge(flows[flows["tfunc_code"] == "C_110"].drop(columns="tfunc_code"),
              on=["provider_code", "month"], how="left")
df["net_flow"] = df["referrals"] - df["completed"]
df["net_flow_3m"] = df.groupby("provider_code")["net_flow"].transform(lambda s: s.rolling(3).mean())

df = df.dropna(subset=["target_delta"])      # drops only each provider's final dev month, as in 07
print(f"cohort rows with a flows return: {df['net_flow'].notna().mean():.1%} of {len(df)} provider-months")

FEATS = ["total_waiting", "pct_gt18", "pct_gt52", "delta_1", "delta_2", "delta_3",
         "month_of_year", "occupancy_ga", "referrals", "completed", "net_flow", "net_flow_3m"]

cohort rows with a flows return: 99.6% of 10431 provider-months


In [5]:
# Goal: the same two training regimes and validation year as 04/07, proven identical.
train_full = df[df["month"] < VAL_START]                              # Apr 2018 - Mar 2024
train_post = df[(df["month"] >= COVID) & (df["month"] < VAL_START)]   # Mar 2020 - Mar 2024
val = df[df["month"] >= VAL_START]                                    # FY 2024-25

assert val["month"].max() < TEST_START       # a sealed-year row leaking in would trip this
assert len(val) == 1407, len(val)            # must be the exact rows 04 and 07 scored
naive_mae = np.abs(val["target_delta"]).mean()
assert round(naive_mae) == 153, naive_mae    # different rows from 04 would trip this

truth = (val["total_waiting"] + val["target_delta"]).to_numpy()   # next month's actual list
print(f"naive on these rows: MAE {naive_mae:,.0f}   (n={len(val)}) - matches 04/07")
print(len(train_full), "train-full,", len(train_post), "train-post,", len(val), "validation rows")

naive on these rows: MAE 153   (n=1407) - matches 04/07
9024 train-full, 6185 train-post, 1407 validation rows


## Transparent references before any trees

(a) **Net-flow identity**: forecast next month's change as this month's net flow — zero fitted parameters. Expected to overshoot (removals are missing from the outflow). (b) **One-parameter linear**: change = a + b·net_flow — the fitted b says how much of a marginal net-flow unit actually survives into the list.

In [6]:
# Goal: does the flow signal help before any machine learning is applied?
def score(pred, label):
    print(f"{label}: MAE {np.abs(truth - pred).mean():,.0f}   "
          f"RMSE {np.sqrt(((truth - pred) ** 2).mean()):,.0f}   (n={len(val)})")

assert int(val["net_flow"].isna().sum()) == 1    # exactly one known gap trips otherwise: R1A Dec 2024
nf = np.where(val["net_flow"].notna(), val["net_flow"], 0.0)
# R1A (C_110 list of 53) filed no Part_1 rows that month - nothing to report is the
# plausible reading; a missing return forecasts zero net flow = persistence for that row

score(val["total_waiting"].to_numpy() + nf, "net-flow identity      ")

for name, tr in [("full-history", train_full), ("post-covid ", train_post)]:
    t = tr.dropna(subset=["net_flow"])       # provider-months without a flows return (0.3% of cohort)
    X = np.column_stack([np.ones(len(t)), t["net_flow"]])
    beta, *_ = np.linalg.lstsq(X, t["target_delta"].to_numpy(), rcond=None)
    score(val["total_waiting"].to_numpy() + beta[0] + beta[1] * nf,
          f"net-flow linear {name}")
    print(f"    fitted: change = {beta[0]:+.1f} {beta[1]:+.2f} * net_flow   (n={len(t)} training rows)")

net-flow identity      : MAE 250   RMSE 453   (n=1407)
net-flow linear full-history: MAE 154   RMSE 328   (n=1407)
    fitted: change = +20.8 +0.08 * net_flow   (n=8983 training rows)
net-flow linear post-covid : MAE 155   RMSE 329   (n=1407)
    fitted: change = +25.3 +0.13 * net_flow   (n=6166 training rows)


## Gradient boosting with flows

In [7]:
# Goal: the 07 delta-GBM with the flow features added - the direct test of whether
# inflow/outflow beat the MAE-153 bar.
preds = {}
for name, tr in [("full-history", train_full), ("post-covid ", train_post)]:
    m = HistGradientBoostingRegressor(random_state=SEED)
    m.fit(tr[FEATS], tr["target_delta"])
    preds[name.strip()] = val["total_waiting"].to_numpy() + m.predict(val[FEATS])
    score(preds[name.strip()], f"gbm-delta+flows {name}")
score(val["total_waiting"].to_numpy(), "naive persistence      ")

gbm-delta+flows full-history: MAE 155   RMSE 335   (n=1407)


gbm-delta+flows post-covid : MAE 158   RMSE 338   (n=1407)
naive persistence      : MAE 153   RMSE 330   (n=1407)


In [8]:
# Goal: permutation importance on the full-history model (the better GBM regime in 07):
# MAE increase on validation when one feature column is shuffled.
m = HistGradientBoostingRegressor(random_state=SEED)
m.fit(train_full[FEATS], train_full["target_delta"])
imp = permutation_importance(m, val[FEATS], val["target_delta"],
                             scoring="neg_mean_absolute_error", random_state=SEED)
print(pd.Series(imp.importances_mean, index=FEATS).sort_values(ascending=False).round(2))

delta_1          7.67
total_waiting    5.33
net_flow_3m      3.67
occupancy_ga     3.45
referrals        2.87
delta_3          2.73
delta_2          2.60
net_flow         2.24
month_of_year    1.79
completed        1.14
pct_gt52         1.03
pct_gt18         0.84
dtype: float64


In [9]:
# Goal: store the validation forecasts so these numbers stay re-readable without a rerun.
out = val[["provider_code", "month"]].copy()
out["target"] = truth
out["pred_naive"] = val["total_waiting"].to_numpy()
out["pred_netflow"] = val["total_waiting"].to_numpy() + nf   # same guarded net flow as the reference
out["pred_gbm_flows_full"] = preds["full-history"]
out["pred_gbm_flows_post"] = preds["post-covid"]
out.to_csv(PROC / "val_forecasts_c110_flows.csv", index=False)
print(out.shape, "->", PROC / "val_forecasts_c110_flows.csv")

(1407, 7) -> ..\data\processed\val_forecasts_c110_flows.csv
